In [259]:
from pathlib import Path

import pandas as pd
import numpy as np
import psycopg

In [260]:
PREDICTIONS_FILE = Path("../data/nq_first_to_100_predictions_full.csv")
DB_NAME = "dailyedge_development"

# Study window established in the previous notebook
STUDY_START = pd.Timestamp("2024-09-01")
STUDY_END = pd.Timestamp("2026-08-21")

# RTH session in Chicago time
RTH_START = "08:30:00"
RTH_END = "15:15:00"

In [261]:
predictions = pd.read_csv(PREDICTIONS_FILE)

predictions["Date"] = pd.to_datetime(predictions["Date"])

predictions.shape, predictions["Date"].min(), predictions["Date"].max()

((508, 9), Timestamp('2024-09-02 00:00:00'), Timestamp('2026-08-21 00:00:00'))

In [262]:
print(predictions.columns.tolist())
print()

print("Bias:")
print(predictions["Bias"].value_counts(dropna=False))

print("\nResult:")
print(predictions["Result"].value_counts(dropna=False))

['Date', 'Day', 'Bias', 'Confidence', 'Auction Direction', 'Context', 'Result', 'Correct', 'Notes']

Bias:
Bias
Long     266
Short    241
NaN        1
Name: count, dtype: int64

Result:
Result
Short      250
Long       231
Invalid     20
Neither      6
NaN          1
Name: count, dtype: int64


In [263]:
query = """
SELECT
    timestamp,
    open,
    high,
    low,
    close,
    volume
FROM CANDLES
WHERE timestamp::date BETWEEN %s AND %s
  AND timestamp::time BETWEEN %s AND %s
ORDER BY timestamp
"""

with psycopg.connect(f"dbname={DB_NAME}") as conn:
    candles = pd.read_sql_query(
        query,
        conn,
        params=(
            STUDY_START.date(),
            STUDY_END.date(),
            RTH_START,
            RTH_END,
        ),
    )

candles.shape, candles["timestamp"].min(), candles["timestamp"].max()

/tmp/ipykernel_18332/167905497.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  candles = pd.read_sql_query(


((202598, 6),
 Timestamp('2024-09-02 08:30:00'),
 Timestamp('2026-08-21 15:15:00'))

In [264]:
prediction_dates = set(predictions["Date"].dropna().dt.date)
candle_dates = set(candles["timestamp"].dt.date)

matched_dates = prediction_dates & candle_dates
prediction_only_dates = prediction_dates - candle_dates
candle_only_dates = candle_dates - prediction_dates

print(f"Prediction rows: {len(predictions)}")
print(f"Candle sessions: {len(candle_dates)}")
print(f"Matched dates: {len(matched_dates)}")
print(f"Prediction-only dates: {sorted(prediction_only_dates)}")
print(f"Candle-only dates: {sorted(candle_only_dates)}")

Prediction rows: 508
Candle sessions: 508
Matched dates: 505
Prediction-only dates: [datetime.date(2025, 1, 9), datetime.date(2026, 4, 3)]
Candle-only dates: [datetime.date(2025, 1, 20), datetime.date(2025, 3, 4), datetime.date(2026, 4, 24)]


In [265]:
candles["Date"] = candles["timestamp"].dt.date
candles["Time"] = candles["timestamp"].dt.time

matched_candles = candles[candles["Date"].isin(matched_dates)]

session_has_close = (
    matched_candles
    .groupby("Date")["Time"]
    .apply(lambda times: pd.Timestamp("15:15:00").time() in set(times))
)

missing_close_dates = session_has_close[~session_has_close].index.tolist()

print(f"Matched sessions: {session_has_close.size}")
print(f"Sessions missing 15:15 close: {len(missing_close_dates)}")

missing_close_dates

Matched sessions: 505
Sessions missing 15:15 close: 18


[datetime.date(2024, 9, 2),
 datetime.date(2024, 11, 28),
 datetime.date(2024, 11, 29),
 datetime.date(2024, 12, 24),
 datetime.date(2025, 2, 17),
 datetime.date(2025, 5, 26),
 datetime.date(2025, 6, 19),
 datetime.date(2025, 7, 3),
 datetime.date(2025, 7, 4),
 datetime.date(2025, 9, 1),
 datetime.date(2025, 11, 27),
 datetime.date(2025, 11, 28),
 datetime.date(2025, 12, 24),
 datetime.date(2026, 1, 19),
 datetime.date(2026, 2, 16),
 datetime.date(2026, 5, 25),
 datetime.date(2026, 6, 19),
 datetime.date(2026, 7, 3)]

In [266]:
valid_dates = matched_dates - set(missing_close_dates)

study_predictions = (
    predictions[predictions["Date"].dt.date.isin(valid_dates)]
    .copy()
    .sort_values("Date")
    .reset_index(drop=True)
)

study_candles = (
    candles[candles["Date"].isin(valid_dates)]
    .copy()
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(f"Prediction rows: {len(study_predictions)}")
print(f"Candle sessions: {study_candles['Date'].nunique()}")
print(f"RTH candles: {len(study_candles)}")

print("\nResult distribution:")
print(study_predictions["Result"].value_counts(dropna=False))

Prediction rows: 487
Candle sessions: 487
RTH candles: 197721

Result distribution:
Result
Short      250
Long       231
Neither      6
Name: count, dtype: int64


In [267]:
WEEKDAY_STRATEGY = {
    "Monday": {
        "stop": 50,
        "target": 100,
        "execution": "BE + Add",
    },
    "Tuesday": {
        "stop": 50,
        "target": 80,
        "execution": "BE + Add",
    },
    "Wednesday": {
        "stop": 75,
        "target": 150,
        "execution": "BE + Add",
    },
    "Thursday": {
        "stop": 65,
        "target": 150,
        "execution": "BE + Add",
    },
    "Friday": {
        "stop": 75,
        "target": 150,
        "execution": "BE + Add",
    },
}

WEEKDAY_STRATEGY

{'Monday': {'stop': 50, 'target': 100, 'execution': 'BE + Add'},
 'Tuesday': {'stop': 50, 'target': 80, 'execution': 'BE + Add'},
 'Wednesday': {'stop': 75, 'target': 150, 'execution': 'BE + Add'},
 'Thursday': {'stop': 65, 'target': 150, 'execution': 'BE + Add'},
 'Friday': {'stop': 75, 'target': 150, 'execution': 'BE + Add'}}

In [268]:
REVERSE_DAYS = {"Thursday"}
REVERSE_BIAS = {"Long": "Short", "Short": "Long"}

In [269]:
def evaluate_one_time_be_add_long(
    day_df,
    stop_distance,
    target_distance
):
    open_price = day_df.iloc[0]["open"]

    initial_stop = open_price - stop_distance
    add_price = open_price + stop_distance
    target_price = open_price + target_distance

    added = False

    for _, candle in day_df.iterrows():

        # Before the add triggers
        if not added:
            stop_hit = candle["low"] <= initial_stop
            add_hit = candle["high"] >= add_price
            target_hit = candle["high"] >= target_price

            # Intrabar ordering cannot be determined
            if stop_hit and (add_hit or target_hit):
                return "Unknown", None

            if target_hit:
                return "Target", target_distance

            if add_hit:
                added = True
                continue

            if stop_hit:
                return "Stopped", -stop_distance

        # After the add:
        # Unit 1 stop = original entry
        # Unit 2 stop = original entry
        # Both target the original target
        else:
            combined_stop = open_price

            stop_hit = candle["low"] <= combined_stop
            target_hit = candle["high"] >= target_price

            # Cannot determine whether target or stop happened first
            if stop_hit and target_hit:
                return "Unknown", None

            if target_hit:
                unit_1_pnl = target_distance
                unit_2_pnl = target_distance - stop_distance

                return "Target", unit_1_pnl + unit_2_pnl

            if stop_hit:
                unit_1_pnl = 0
                unit_2_pnl = -stop_distance

                return "Stopped", unit_1_pnl + unit_2_pnl

    # Session ends before either stop or target
    final_close = day_df.iloc[-1]["close"]

    if not added:
        pnl = final_close - open_price
    else:
        unit_1_pnl = final_close - open_price
        unit_2_pnl = final_close - add_price
        pnl = unit_1_pnl + unit_2_pnl

    return "Close", pnl

In [270]:
def evaluate_one_time_be_add_short(
    day_df,
    stop_distance,
    target_distance
):
    open_price = day_df.iloc[0]["open"]

    initial_stop = open_price + stop_distance
    add_price = open_price - stop_distance
    target_price = open_price - target_distance

    added = False

    for _, candle in day_df.iterrows():

        # Before the add triggers
        if not added:
            stop_hit = candle["high"] >= initial_stop
            add_hit = candle["low"] <= add_price
            target_hit = candle["low"] <= target_price

            # Intrabar ordering cannot be determined
            if stop_hit and (add_hit or target_hit):
                return "Unknown", None

            if target_hit:
                return "Target", target_distance

            if add_hit:
                added = True
                continue

            if stop_hit:
                return "Stopped", -stop_distance

        # After the add
        else:
            combined_stop = open_price

            stop_hit = candle["high"] >= combined_stop
            target_hit = candle["low"] <= target_price

            if stop_hit and target_hit:
                return "Unknown", None

            if target_hit:
                unit_1_pnl = target_distance
                unit_2_pnl = target_distance - stop_distance

                return "Target", unit_1_pnl + unit_2_pnl

            if stop_hit:
                unit_1_pnl = 0
                unit_2_pnl = -stop_distance

                return "Stopped", unit_1_pnl + unit_2_pnl

    final_close = day_df.iloc[-1]["close"]

    if not added:
        pnl = open_price - final_close
    else:
        unit_1_pnl = open_price - final_close
        unit_2_pnl = add_price - final_close
        pnl = unit_1_pnl + unit_2_pnl

    return "Close", pnl

In [271]:
strategy_rows = []

for _, prediction in study_predictions.iterrows():
    date = prediction["Date"].date()
    weekday = prediction["Day"]
    predicted_bias = prediction["Bias"]

    bias = (
        REVERSE_BIAS.get(predicted_bias, predicted_bias)
        if weekday in REVERSE_DAYS
        else predicted_bias
    )

    config = WEEKDAY_STRATEGY[weekday]

    if config is None:
        strategy_rows.append({
            "Date": date,
            "Day": weekday,
            "Predicted_Bias": predicted_bias,
            "Bias": bias,
            "Stop": None,
            "Target": None,
            "Outcome": "Skipped",
            "PnL": None,
        })
        continue

    stop_distance = config["stop"]
    target_distance = config["target"]

    day_df = study_candles[
        study_candles["Date"] == date
    ]

    if bias == "Long":
        outcome, pnl = evaluate_one_time_be_add_long(
            day_df,
            stop_distance,
            target_distance,
        )
    else:
        outcome, pnl = evaluate_one_time_be_add_short(
            day_df,
            stop_distance,
            target_distance,
        )

    strategy_rows.append({
        "Date": date,
        "Day": weekday,
        "Predicted_Bias": predicted_bias,
        "Bias": bias,
        "Stop": stop_distance,
        "Target": target_distance,
        "Outcome": outcome,
        "PnL": pnl,
    })

strategy_results = pd.DataFrame(strategy_rows)

strategy_results["Outcome"].value_counts(dropna=False)

Outcome
Stopped    307
Target     170
Close        8
Unknown      2
Name: count, dtype: int64

In [272]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

weekday_validation = (
    strategy_results
    .groupby("Day", sort=False)
    .agg(
        Sessions=("Date", "count"),
        Stop=("Stop", "first"),
        Target=("Target", "first"),
        Skipped=("Outcome", lambda x: (x == "Skipped").sum()),
        Unknown=("Outcome", lambda x: (x == "Unknown").sum()),
    )
    .reindex(weekday_order)
)

weekday_validation

,Sessions,Stop,Target,Skipped,Unknown
Day,,,,,
Monday,95,50,100,0,0
Tuesday,101,50,80,0,2
Wednesday,100,75,150,0,0
Thursday,96,65,150,0,0
Friday,95,75,150,0,0


In [273]:
traded = strategy_results[
    strategy_results["Outcome"] != "Skipped"
].copy()

resolved = traded[
    traded["Outcome"] != "Unknown"
].copy()

attempted_sessions = len(traded)
resolved_trades = len(resolved)
unknown_trades = (traded["Outcome"] == "Unknown").sum()
skipped_thursdays = (strategy_results["Outcome"] == "Skipped").sum()

total_pnl = resolved["PnL"].sum()
average_pnl = resolved["PnL"].mean()
median_pnl = resolved["PnL"].median()

winning_trades = (resolved["PnL"] > 0).sum()
losing_trades = (resolved["PnL"] < 0).sum()
breakeven_trades = (resolved["PnL"] == 0).sum()

win_rate = winning_trades / resolved_trades

average_winner = resolved.loc[resolved["PnL"] > 0, "PnL"].mean()
average_loser = resolved.loc[resolved["PnL"] < 0, "PnL"].mean()

print(f"Attempted trading sessions: {attempted_sessions}")
print(f"Resolved trades: {resolved_trades}")
print(f"Unknown trades: {unknown_trades}")
print(f"Skipped Thursday sessions: {skipped_thursdays}")
print()
print(f"Total P&L: {total_pnl:.2f} points")
print(f"Average P&L: {average_pnl:.2f} points/trade")
print(f"Median P&L: {median_pnl:.2f} points")
print()
print(f"Winning trades: {winning_trades}")
print(f"Losing trades: {losing_trades}")
print(f"Breakeven trades: {breakeven_trades}")
print(f"Win rate: {win_rate:.2%}")
print()
print(f"Average winner: {average_winner:.2f} points")
print(f"Average loser: {average_loser:.2f} points")

Attempted trading sessions: 487
Resolved trades: 485
Unknown trades: 2
Skipped Thursday sessions: 0

Total P&L: 12925.10 points
Average P&L: 26.65 points/trade
Median P&L: -50.00 points

Winning trades: 177
Losing trades: 308
Breakeven trades: 0
Win rate: 36.49%

Average winner: 182.66 points
Average loser: -63.01 points


In [274]:
gross_profit = resolved.loc[resolved["PnL"] > 0, "PnL"].sum()
gross_loss = abs(resolved.loc[resolved["PnL"] < 0, "PnL"].sum())

profit_factor = gross_profit / gross_loss

resolved = resolved.sort_values("Date").reset_index(drop=True)
resolved["Cumulative_PnL"] = resolved["PnL"].cumsum()

running_peak = resolved["Cumulative_PnL"].cummax()
drawdown = resolved["Cumulative_PnL"] - running_peak

max_drawdown = drawdown.min()

is_loss = resolved["PnL"] < 0
is_win = resolved["PnL"] > 0
streak_id = (is_loss != is_loss.shift()).cumsum()

streaks = (
    resolved
    .groupby(streak_id)
    .agg(
        Start=("Date", "first"),
        End=("Date", "last"),
        Length=("PnL", "size"),
        Is_Loss=("PnL", lambda x: (x < 0).all()),
        Is_Win=("PnL", lambda x: (x > 0).all()),
    )
)

losing_streaks = streaks[streaks["Is_Loss"]]
winning_streaks = streaks[streaks["Is_Win"]]

longest_losing = losing_streaks.loc[losing_streaks["Length"].idxmax()]
longest_winning = winning_streaks.loc[winning_streaks["Length"].idxmax()]

max_winning_streak = longest_winning["Length"]
max_losing_streak = longest_losing["Length"]

avg_winning_streak = winning_streaks["Length"].mean()
avg_losing_streak = losing_streaks["Length"].mean()

print(f"Gross profit: {gross_profit:.2f} points")
print(f"Gross loss: {gross_loss:.2f} points")
print(f"Profit factor: {profit_factor:.2f}")
print()
print(f"Maximum cumulative drawdown: {max_drawdown:.2f} points")
print()
print(f"Maximum winning streak: {max_winning_streak} ({longest_winning['Start']} to {longest_winning['End']})")
print(f"Maximum losing streak: {max_losing_streak} ({longest_losing['Start']} to {longest_losing['End']})")
print(f"Average winning streak: {avg_winning_streak:.2f} trades")
print(f"Average losing streak: {avg_losing_streak:.2f} trades")

Gross profit: 32331.68 points
Gross loss: 19406.58 points
Profit factor: 1.67

Maximum cumulative drawdown: -1271.76 points

Maximum winning streak: 4 (2024-10-16 to 2024-10-21)
Maximum losing streak: 9 (2025-05-16 to 2025-05-29)
Average winning streak: 1.57 trades
Average losing streak: 2.75 trades


In [275]:
resolved["Week"] = resolved["Date"].apply(
    lambda d: pd.Timestamp(d) - pd.Timedelta(days=pd.Timestamp(d).weekday())
)

weekly_pnl = resolved.groupby("Week")["PnL"].sum()

weekly_expectancy = weekly_pnl.mean()
weekly_win_rate = (weekly_pnl > 0).mean()

print(f"Weeks traded: {len(weekly_pnl)}")
print(f"Weekly expectancy: {weekly_expectancy:.2f} points/week")
print(f"Weekly win rate: {weekly_win_rate:.2%}")
print(f"Best week: {weekly_pnl.max():.2f}")
print(f"Worst week: {weekly_pnl.min():.2f}")

weekly_pnl.describe()

Weeks traded: 103
Weekly expectancy: 125.49 points/week
Weekly win rate: 61.17%
Best week: 785.00
Worst week: -315.00


count    103.000000
mean     125.486438
std      267.991325
min     -315.000000
25%      -15.000000
50%      145.000000
75%      285.000000
max      785.000000
Name: PnL, dtype: float64